# W7-4 Robustness: Q11 replenishment comparison with lead-time variation
**Owner:** Ibrahim Khalil Umar · **Week 7, W7-4** · reviewer: Abdibasid Mohamed Ali

**Question.** Q11 compared the existing reorder levels (policy A) with a demand-based trigger (policy B) using one constant lead time per pair. Lead times in the purchase orders are not constant (7 to 35 days). Does the demand-based trigger still beat the existing levels when each order's lead time is drawn from the pair's own history?

> **Every stock figure here rests on one assumption: that the sales scale is the right reference for stock (Q10 unresolved).** No holding or ordering cost data exists, so no policy is called cost-optimal.

In [1]:
import os, pathlib, math, time
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42; RESAMPLES = 5000; REPS = 1000
HERE = pathlib.Path.cwd()
def find_merged():
    env = os.environ.get("CADETX_MERGED")
    if env and pathlib.Path(env).exists():
        return pathlib.Path(env)
    for p in [HERE, *HERE.parents]:
        c = p / "week-02" / "reports" / "week_02" / "merged"
        if c.exists():
            return c
    raise SystemExit("Merged datasets not found: set CADETX_MERGED or run inside the repository checkout.")
MERGED = find_merged()
OUT = HERE / "w74_outputs"; CH = HERE / "w74_charts"; OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
LABEL = "Sales scale assumed (Q10 unresolved); results rest on this assumption"
_COMMON = ("lost sales, no backorder; start stock = opening_stock; one historical demand path; policy B reorder points are estimated from the same history they are tested on (in-sample); "
           "no holding or ordering cost, so no policy is called cost-optimal")
ASSUMPTIONS = ("each order's lead time drawn independently from the pair's received-PO lead times (orders may cross); " + _COMMON)
ASSUMPTIONS_CONST = ("one constant lead time per pair (the rounded mean, 17 to 19 days), as in Weeks 5 and 6; " + _COMMON)
GREY, INK, INK2, GRID, SURF = "#8A8F98", "#16213A", "#4A5163", "#E3E5EA", "#FFFFFF"
C_A, C_B, C_BLT = "#C8791A", "#1B6B66", "#16213A"
checks = []
def check(name, expected, actual, ok):
    checks.append(dict(Check_Item=name, Expected=expected, Actual=actual, Status="PASS" if ok else "FAIL"))
    assert ok, f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)

K = ["product_id", "branch_id"]
L = pd.read_csv(MERGED / "stock_ledger_enriched.csv", low_memory=False, parse_dates=["movement_date"])
I = pd.read_csv(MERGED / "inventory_with_ledger.csv")
P = pd.read_csv(MERGED / "purchase_orders_full.csv", low_memory=False,
                usecols=["po_id", "product_id", "branch_id", "po_status", "lead_time_actual_days"])
print("data folder:", MERGED)
tot = L.groupby("movement_type").quantity.sum()
check("ledger IN is about 18.1 times OUT (Q10 reproduces)", "18.13 +/- 0.1", round(tot["IN"] / tot["OUT"], 2), abs(tot["IN"] / tot["OUT"] - 18.13) < 0.1)
check("180 product-branch pairs", "180", str(len(I)), len(I) == 180)

data folder: /tmp/claude-0/-home-claude/6bcba5f3-dda2-517e-bd95-e2227d15c22a/scratchpad/w7/final2/week-02/reports/week_02/merged


## 1. Demand and lead-time history

In [2]:
O = L[L.movement_type == "OUT"]
dates = pd.date_range(O.movement_date.min(), O.movement_date.max(), freq="D")
pidx = pd.MultiIndex.from_frame(I[K])
dem = O.groupby(K + ["movement_date"]).quantity.sum().unstack("movement_date").reindex(columns=dates, fill_value=0).fillna(0).reindex(pidx, fill_value=0)
D = dem.values.astype(float); T = D.shape[1]; NP = len(I)
check("demand matrix total equals ledger OUT total", str(int(O.quantity.sum())), str(int(D.sum())), int(D.sum()) == int(O.quantity.sum()))
check("full daily calendar of 2,203 days", "(180, 2203)", str(D.shape), D.shape == (180, 2203))

Pr = P[P.po_status.eq("Received") & P.lead_time_actual_days.notna()]
grp = Pr.groupby(K).lead_time_actual_days
emp = [grp.get_group(tuple(k)).values.astype(int) for k in pidx]
cnt = np.array([len(e) for e in emp]); W = cnt.max()
E = np.zeros((NP, W), int)
for i, e in enumerate(emp): E[i, :len(e)] = e
ltm = np.array([e.mean() for e in emp]); lts = np.array([e.std(ddof=1) for e in emp])
Lead = np.maximum(1, np.round(ltm)).astype(int)          # the constant lead time used in Weeks 5 and 6
mu = D.mean(axis=1); sd = D.std(axis=1, ddof=1)
allL = Pr.lead_time_actual_days.values
ltq = pd.DataFrame({"product_id": I.product_id, "branch_id": I.branch_id, "received_po_lines": cnt,
                    "lead_mean": ltm.round(2), "lead_sd": lts.round(2), "lead_min": [e.min() for e in emp],
                    "lead_p05": [np.percentile(e, 5) for e in emp], "lead_median": [np.median(e) for e in emp],
                    "lead_p95": [np.percentile(e, 95) for e in emp], "lead_max": [e.max() for e in emp],
                    "constant_lead_used_in_week6": Lead})
ltq["label"] = LABEL; ltq.to_csv(OUT / "w74_lead_time_by_pair.csv", index=False)
print(f"received PO lines {len(allL):,}; lead time mean {allL.mean():.2f} sd {allL.std(ddof=1):.2f} min {allL.min()} max {allL.max()}")
check("every pair has at least 600 received PO lines with a lead time", ">= 600", str(cnt.min()), cnt.min() >= 600)
check("pooled lead time ranges 7 to 35 days", "7 to 35", f"{allL.min()} to {allL.max()}", allL.min() == 7 and allL.max() == 35)
check("pooled lead-time SD is about 6.7 days (6.5 to 6.9)", "6.5 to 6.9", round(allL.std(ddof=1), 2), 6.5 <= allL.std(ddof=1) <= 6.9)
check("every pair's lead-time SD is at least 6 days (variation is not a few pairs)", ">= 6", round(lts.min(), 2), lts.min() >= 6)
check("constant lead times are the rounded pair means (17 to 19 days)", "17 to 19", f"{Lead.min()} to {Lead.max()}", Lead.min() >= 17 and Lead.max() <= 19)

received PO lines 140,077; lead time mean 17.90 sd 6.67 min 7.0 max 35.0


## 2. Simulator (all pairs and replications run together)
Same rules as Weeks 5 and 6: daily review; when on hand plus on order is at or below `s`, order up to `S`; lost sales; start stock = `opening_stock`. The only change is that each order's lead time can be drawn from the pair's received-PO lead times. The same random draws are used for every policy (common random numbers), so differences between policies are not sampling noise.

In [3]:
def simulate(s, S, reps, seed, const=False, U=None, pairs=None):
    """Vectorised over pairs x reps. const=True uses the Week 6 constant lead time.
    U (optional) is a (N, T) matrix of uniforms for the lead-time draw, used only for the cross-check."""
    pairs = np.arange(NP) if pairs is None else np.asarray(pairs)
    n = len(pairs); N = n * reps
    rng = np.random.default_rng(seed)
    s_ = np.tile(s, reps); S_ = np.tile(S, reps); pr = np.tile(pairs, reps)
    B = 64; pipe = np.zeros((N, B)); onhand = np.tile(I.opening_stock.values.astype(float)[pairs], reps)
    onorder = np.zeros(N); unmet = np.zeros(N); sdays = np.zeros(N); area = np.zeros(N); orders = np.zeros(N)
    rows = np.arange(N); cn = cnt[pr]
    for t in range(T):
        sl = t % B; a = pipe[:, sl].copy(); pipe[:, sl] = 0
        onhand += a; onorder -= a
        d = np.tile(D[pairs, t], reps); sold = np.minimum(d, onhand)
        unmet += d - sold; sdays += (d > sold); onhand -= sold
        pos = onhand + onorder; q = S_ - pos; m = (pos <= s_) & (q > 0)
        if const:
            u = None
        else:
            u = U[:, t] if U is not None else rng.random(N)
        if m.any():
            r = rows[m]
            lead = Lead[pr[r]] if const else E[pr[r], np.minimum((u[r] * cn[r]).astype(int), cn[r] - 1)]
            pipe[r, (t + lead) % B] += q[r]; onorder[r] += q[r]; orders[r] += 1
        area += onhand
    f = lambda x: x.reshape(reps, n)
    return dict(unmet=f(unmet), stockout_days=f(sdays), avg_on_hand=f(area / T), orders=f(orders))

def simulate_scalar(i, s, S, U_row):
    """Plain day-by-day loop for one pair, same draws as the vectorised run (cross-check)."""
    pipe = {}; onhand = float(I.opening_stock.iloc[i]); onorder = 0.0; unmet = 0.0; area = 0.0; orders = 0
    for t in range(T):
        a = pipe.pop(t, 0.0); onhand += a; onorder -= a
        d = D[i, t]; sold = min(d, onhand); unmet += d - sold; onhand -= sold
        pos = onhand + onorder
        if pos <= s and S - pos > 0:
            lead = E[i, min(int(U_row[t] * cnt[i]), cnt[i] - 1)]
            pipe[t + lead] = pipe.get(t + lead, 0.0) + (S - pos); onorder += S - pos; orders += 1
        area += onhand
    return unmet, area / T, orders

sA = I.reorder_level.values.astype(float); SA = I.max_stock.values.astype(float); gap = SA - sA
def rop_mean(z):            # Week 5/6 trigger: constant mean lead time
    return np.ceil(mu * Lead + z * sd * np.sqrt(Lead))
def rop_var(z):             # lead-time-aware trigger: adds the variance of lead-time demand
    return np.ceil(mu * ltm + z * np.sqrt(ltm * sd**2 + mu**2 * lts**2))
POL = {"A existing (reorder_level, max_stock)": (sA, SA)}
for z, nm in [(1.28, "90%"), (1.645, "95%"), (2.33, "99%")]:
    s = rop_mean(z); POL[f"B {nm} (mean-lead trigger)"] = (s, s + gap)
for z, nm in [(1.28, "90%"), (1.645, "95%"), (2.33, "99%")]:
    s = rop_var(z); POL[f"B-LT {nm} (lead-time-aware trigger)"] = (s, s + gap)

# cross-check the vectorised simulator against a plain loop on 5 pairs, same random draws
chk_pairs = [0, 37, 71, 120, 179]
U = np.random.default_rng(SEED).random((5, T))
s95 = rop_mean(1.645)
for nm, (s, S) in [("A", (sA, SA)), ("B95", (s95, s95 + gap))]:
    v = simulate(s[chk_pairs], S[chk_pairs], 1, SEED, U=U, pairs=chk_pairs)
    ok = True
    for k, i in enumerate(chk_pairs):
        u_, av_, o_ = simulate_scalar(i, s[i], S[i], U[k])
        ok &= abs(u_ - v["unmet"][0, k]) < 1e-6 and abs(av_ - v["avg_on_hand"][0, k]) < 1e-6 and o_ == v["orders"][0, k]
    check(f"vectorised simulator equals a plain day-by-day loop on 5 pairs (policy {nm})", "identical", "identical" if ok else "differs", ok)

## 3. Reproduce the constant-lead result, then run with lead-time variation

In [4]:
tot_dem = D.sum(axis=1); cost = I.unit_cost.values.astype(float)
def portfolio(r):
    fill_rep = 1 - r["unmet"].sum(axis=1) / tot_dem.sum()           # one value per replication
    pair_fill = 1 - r["unmet"] / tot_dem                             # reps x pairs
    return fill_rep, pair_fill
rows = []; PAIRFILL = {}; REPFILL = {}; VAL = {}; PAIRDF = []
for case, const, reps in [("constant lead time (Week 6)", True, 1), ("lead time varies (drawn per order)", False, REPS)]:
    t0 = time.time()
    for name, (s, S) in POL.items():
        if const and name.startswith("B-LT"): continue
        r = simulate(s, S, reps, SEED, const=const)
        fr, pf = portfolio(r)
        val_rep = (r["avg_on_hand"] * cost).sum(axis=1)
        PAIRFILL[(case, name)] = pf.mean(axis=0); REPFILL[(case, name)] = fr; VAL[(case, name)] = val_rep
        pair_mean_fill = pf.mean(axis=0)
        rows.append(dict(lead_time_case=case, policy=name, replications=reps,
                         fill_rate=fr.mean(), fill_rate_p2_5=np.percentile(fr, 2.5), fill_rate_p97_5=np.percentile(fr, 97.5),
                         stockout_day_share=r["stockout_days"].sum(axis=1).mean() / (NP * T),
                         avg_on_hand_per_pair=r["avg_on_hand"].mean(),
                         avg_stock_value_at_cost=val_rep.mean(), stock_value_p2_5=np.percentile(val_rep, 2.5), stock_value_p97_5=np.percentile(val_rep, 97.5),
                         orders_placed=r["orders"].sum(axis=1).mean(),
                         pairs_below_95_mean_fill=int((pair_mean_fill < .95).sum()),
                         pairs_below_95_mean_across_reps=float((pf < .95).sum(axis=1).mean())))
        PAIRDF.append(pd.DataFrame({"product_id": I.product_id, "branch_id": I.branch_id, "lead_time_case": case, "policy": name,
                                    "s": s, "S": S, "fill_rate": pair_mean_fill,
                                    "avg_on_hand": r["avg_on_hand"].mean(axis=0), "orders_placed": r["orders"].mean(axis=0),
                                    "stockout_days": r["stockout_days"].mean(axis=0)}))
    print(case, "done in", round(time.time() - t0, 1), "s")
SUM = pd.DataFrame(rows)
SUM["label"] = LABEL; SUM["assumptions"] = np.where(SUM.lead_time_case.str.startswith("constant"), ASSUMPTIONS_CONST, ASSUMPTIONS)
SUM.to_csv(OUT / "w74_policy_summary.csv", index=False)
PD_ = pd.concat(PAIRDF, ignore_index=True); PD_["label"] = LABEL; PD_["assumptions"] = np.where(PD_.lead_time_case.str.startswith("constant"), ASSUMPTIONS_CONST, ASSUMPTIONS)
PD_.to_csv(OUT / "w74_pair_results.csv", index=False)
show = SUM[["lead_time_case", "policy", "fill_rate", "fill_rate_p2_5", "fill_rate_p97_5", "avg_stock_value_at_cost", "orders_placed", "pairs_below_95_mean_fill"]].copy()
print(show.round(4).to_string(index=False))

c0 = SUM[SUM.lead_time_case.str.startswith("constant")].set_index("policy")
check("constant-lead rerun reproduces Week 6 fill rates (A 94.3%, B 98.9 / 99.4 / 99.8%)", "0.943 / 0.989 / 0.994 / 0.998",
      " / ".join(f"{c0.fill_rate.iloc[i]:.3f}" for i in range(4)),
      all(abs(c0.fill_rate.iloc[i] - e) < 0.0006 for i, e in enumerate([.9426, .9891, .9939, .9981])))
check("constant-lead rerun reproduces Week 6 orders (4,457 / 4,804 / 4,839 / 4,872)", "4457 / 4804 / 4839 / 4872",
      " / ".join(str(int(x)) for x in c0.orders_placed), [int(x) for x in c0.orders_placed] == [4457, 4804, 4839, 4872])
check("constant-lead rerun reproduces Week 6 stock value (A 392.6M, B 441.4M / 468.3M / 519.1M)", "392.6 / 441.4 / 468.3 / 519.1",
      " / ".join(f"{x/1e6:.1f}" for x in c0.avg_stock_value_at_cost), all(abs(x/1e6 - e) < 0.06 for x, e in zip(c0.avg_stock_value_at_cost, [392.6, 441.4, 468.3, 519.1])))
check("constant-lead rerun reproduces 72 pairs below 95% under policy A", "72", str(int(c0.pairs_below_95_mean_fill.iloc[0])), int(c0.pairs_below_95_mean_fill.iloc[0]) == 72)
check("1,000 replications for the varying-lead case, seed 42", "1000", str(int(SUM[SUM.lead_time_case.str.startswith("lead")].replications.iloc[0])), REPS == 1000 and SEED == 42)
check("fill rate lies between 0 and 1 for every policy and pair", "0 to 1", "ok", PD_.fill_rate.between(0, 1).all())
check("7 policies x 180 pairs in the varying-lead case, 4 x 180 in the constant case", "1260 + 720 rows", str(len(PD_)), len(PD_) == 7 * 180 + 4 * 180)
v = SUM[SUM.lead_time_case.str.startswith("lead")].set_index("policy")
check("higher service factor never lowers fill (B 90 <= 95 <= 99; B-LT likewise)", "monotone", "monotone",
      v.fill_rate.iloc[1] < v.fill_rate.iloc[2] < v.fill_rate.iloc[3] and v.fill_rate.iloc[4] < v.fill_rate.iloc[5] < v.fill_rate.iloc[6])

# simulated lead times have the right mean
rng = np.random.default_rng(SEED); draw = E[np.repeat(np.arange(NP), 4000), np.minimum((rng.random(NP * 4000) * np.repeat(cnt, 4000)).astype(int), np.repeat(cnt, 4000) - 1)]
check("drawn lead times average the pair means (within 0.05 day overall)", "within 0.05", round(abs(draw.mean() - np.average(ltm, weights=np.ones(NP))), 3), abs(draw.mean() - ltm.mean()) < 0.05)

constant lead time (Week 6) done in 0.2 s
lead time varies (drawn per order) done in 101.7 s
                    lead_time_case                                policy  fill_rate  fill_rate_p2_5  fill_rate_p97_5  avg_stock_value_at_cost  orders_placed  pairs_below_95_mean_fill
       constant lead time (Week 6) A existing (reorder_level, max_stock)     0.9426          0.9426           0.9426             3.925898e+08       4457.000                        72
       constant lead time (Week 6)             B 90% (mean-lead trigger)     0.9891          0.9891           0.9891             4.413722e+08       4804.000                         0
       constant lead time (Week 6)             B 95% (mean-lead trigger)     0.9939          0.9939           0.9939             4.683394e+08       4839.000                         0
       constant lead time (Week 6)             B 99% (mean-lead trigger)     0.9981          0.9981           0.9981             5.190848e+08       4872.000                   

## 4. Does the demand-based trigger still beat the existing levels?
Paired differences against policy A under the same draws. Replication spread is the 2.5th to 97.5th percentile across 1,000 lead-time paths; the pair-level interval is a bootstrap over the 180 pairs (5,000 resamples, seed 42) of each pair's mean fill-rate difference.

In [5]:
rng = np.random.default_rng(SEED)
case = "lead time varies (drawn per order)"
A = "A existing (reorder_level, max_stock)"
bi = rng.integers(0, NP, size=(RESAMPLES, NP))                     # same resamples for every comparison
rows = []
for name in [n for n in POL if n != A]:
    diff_pair = PAIRFILL[(case, name)] - PAIRFILL[(case, A)]
    boot = diff_pair[bi].mean(axis=1)
    diff_rep = REPFILL[(case, name)] - REPFILL[(case, A)]
    extra_val = VAL[(case, name)] / VAL[(case, A)] - 1
    rows.append(dict(policy=name, vs="A existing",
                     portfolio_fill_gain_pts=100 * diff_rep.mean(), gain_p2_5_pts=100 * np.percentile(diff_rep, 2.5), gain_p97_5_pts=100 * np.percentile(diff_rep, 97.5),
                     share_of_replications_B_beats_A=float((diff_rep > 0).mean()),
                     mean_pair_fill_gain_pts=100 * diff_pair.mean(), pair_boot_ci_low_pts=100 * np.percentile(boot, 2.5), pair_boot_ci_high_pts=100 * np.percentile(boot, 97.5),
                     pairs_where_B_fill_higher=int((diff_pair > 0).sum()), pairs_where_B_fill_lower=int((diff_pair < 0).sum()),
                     extra_stock_value_pct=100 * extra_val.mean()))
DIFF = pd.DataFrame(rows); DIFF["label"] = LABEL; DIFF["assumptions"] = ASSUMPTIONS
DIFF.to_csv(OUT / "w74_comparison_vs_A.csv", index=False)
print(DIFF.drop(columns=["label", "assumptions"]).round(3).to_string(index=False))

pf_A = PAIRFILL[(case, A)]
check("5,000 bootstrap resamples, seed 42", "5000", str(bi.shape[0]), bi.shape[0] == RESAMPLES)
check("under varying lead times every B variant has a higher portfolio fill than A in 100% of replications", "1.0", str(DIFF.share_of_replications_B_beats_A.min()), (DIFF.share_of_replications_B_beats_A == 1.0).all())
check("pair-level bootstrap interval for the fill gain excludes zero for every B variant", "low > 0", str(round(DIFF.pair_boot_ci_low_pts.min(), 3)), (DIFF.pair_boot_ci_low_pts > 0).all())
check("lead-time variation lowers fill under A and under every B (constant -> varying)", "all lower", "all lower",
      all(v.loc[n, "fill_rate"] < c0.loc[n, "fill_rate"] for n in c0.index))

                            policy         vs  portfolio_fill_gain_pts  gain_p2_5_pts  gain_p97_5_pts  share_of_replications_B_beats_A  mean_pair_fill_gain_pts  pair_boot_ci_low_pts  pair_boot_ci_high_pts  pairs_where_B_fill_higher  pairs_where_B_fill_lower  extra_stock_value_pct
         B 90% (mean-lead trigger) A existing                    4.538          4.322           4.778                              1.0                    4.359                 3.603                  5.138                        141                        37                 12.584
         B 95% (mean-lead trigger) A existing                    5.172          4.949           5.402                              1.0                    4.985                 4.189                  5.804                        159                        20                 18.856
         B 99% (mean-lead trigger) A existing                    5.889          5.686           6.109                              1.0                    5.6

## 5. Where the demand-based trigger falls short, and what it costs

In [6]:
pair = PD_[PD_.lead_time_case == case]
tab = (pair.assign(below95=pair.fill_rate < .95).groupby("policy").agg(pairs_below_95=("below95", "sum"), worst_pair_fill=("fill_rate", "min"),
        median_pair_fill=("fill_rate", "median")).reset_index())
tab["label"] = LABEL; tab.to_csv(OUT / "w74_pairs_below_95.csv", index=False)
print(tab.round(4).to_string(index=False))
worstB = pair[pair.policy.str.startswith("B 95")].sort_values("fill_rate").head(5)[["product_id", "branch_id", "fill_rate"]]
print("lowest five pairs under B 95%:\n", worstB.round(4).to_string(index=False))
A_below = int(tab.set_index("policy").loc[A, "pairs_below_95"]); B90_below = int(tab[tab.policy.str.startswith("B 90")].pairs_below_95.iloc[0])
check("existing levels leave more pairs below 95% fill than any B variant under varying lead times", "A has most", f"A {A_below}, max B {int(tab[tab.policy != A].pairs_below_95.max())}", A_below > tab[tab.policy != A].pairs_below_95.max())

# overlap check: how much of each policy's stock value is above A
trade = SUM[SUM.lead_time_case.str.startswith("lead")][["policy", "fill_rate", "avg_stock_value_at_cost"]].copy()
trade["extra_value_vs_A_M"] = (trade.avg_stock_value_at_cost - trade.avg_stock_value_at_cost.iloc[0]) / 1e6
print(trade.round(4).to_string(index=False))
# does a lead-time-aware trigger buy more fill than raising z on the mean-lead trigger? compare at similar stock value
b = trade.set_index("policy")
print("B-LT 90% value", round(b.loc["B-LT 90% (lead-time-aware trigger)", "avg_stock_value_at_cost"]/1e6, 1), "fill", round(b.loc["B-LT 90% (lead-time-aware trigger)", "fill_rate"], 4),
      "| B 95% value", round(b.loc["B 95% (mean-lead trigger)", "avg_stock_value_at_cost"]/1e6, 1), "fill", round(b.loc["B 95% (mean-lead trigger)", "fill_rate"], 4))

                               policy  pairs_below_95  worst_pair_fill  median_pair_fill                                                                 label
A existing (reorder_level, max_stock)              81           0.7091            0.9615 Sales scale assumed (Q10 unresolved); results rest on this assumption
            B 90% (mean-lead trigger)               1           0.9467            0.9834 Sales scale assumed (Q10 unresolved); results rest on this assumption
            B 95% (mean-lead trigger)               0           0.9653            0.9891 Sales scale assumed (Q10 unresolved); results rest on this assumption
            B 99% (mean-lead trigger)               0           0.9829            0.9957 Sales scale assumed (Q10 unresolved); results rest on this assumption
   B-LT 90% (lead-time-aware trigger)               0           0.9644            0.9877 Sales scale assumed (Q10 unresolved); results rest on this assumption
   B-LT 95% (lead-time-aware trigger)         

## 6. Charts

In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax[0].hist(allL, bins=np.arange(6.5, 36.5, 1), color=C_B, edgecolor="white")
ax[0].axvline(allL.mean(), color=C_A, lw=2); ax[0].text(allL.mean() + .4, ax[0].get_ylim()[1] * .92, f"mean {allL.mean():.1f} days\n(the constant used in Weeks 5 and 6)", color=C_A, fontsize=9, va="top")
ax[0].set_xlabel("Actual lead time of a received purchase-order line, days"); ax[0].set_ylabel("Purchase-order lines"); ax[0].set_title("Lead times run from 7 to 35 days (SD 6.7)")
ax[1].scatter(ltm, lts, s=14, color=INK, alpha=.7); ax[1].set_xlabel("Pair mean lead time, days"); ax[1].set_ylabel("Pair lead-time SD, days")
ax[1].set_xlim(16, 20); ax[1].set_ylim(0, 8); ax[1].set_title("Every one of the 180 pairs has the same spread")
for a in ax: style(a)
fig.text(.5, -.02, "Received purchase-order lines only | " + LABEL, ha="center", fontsize=8.5, color=INK2)
plt.tight_layout(); plt.savefig(CH / "w74_01_lead_time_distribution.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()

names = list(POL.keys()); short = ["A existing", "B 90%", "B 95%", "B 99%", "B-LT 90%", "B-LT 95%", "B-LT 99%"]
cols = [C_A, C_B, C_B, C_B, C_BLT, C_BLT, C_BLT]
fig, ax = plt.subplots(figsize=(11.5, 4.9))
w = .38; x = np.arange(len(names))
cv = [c0.fill_rate.get(n, np.nan) * 100 for n in names]
vv = [v.fill_rate[n] * 100 for n in names]; lo = [v.fill_rate_p2_5[n] * 100 for n in names]; hi = [v.fill_rate_p97_5[n] * 100 for n in names]
ax.bar(x - w / 2, cv, w, color=GRID, edgecolor=GREY, label="Constant lead time (Week 6)")
ax.bar(x + w / 2, vv, w, color=cols, label="Lead time varies (1,000 paths)")
ax.errorbar(x + w / 2, vv, yerr=[np.array(vv) - np.array(lo), np.array(hi) - np.array(vv)], fmt="none", ecolor=INK, capsize=3, lw=1)
for i, (a_, b_) in enumerate(zip(cv, vv)):
    if not np.isnan(a_): ax.text(i - w / 2, a_ + .15, f"{a_:.1f}", ha="center", fontsize=8, color=INK2)
    ax.text(i + w / 2, hi[i] + .15, f"{b_:.1f}", ha="center", fontsize=8.5, color=INK, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(short); ax.set_ylim(90, 102.2); ax.set_ylabel("Fill rate, % of demand met")
ax.set_title("The demand-based trigger still beats the existing levels with variable lead times"); ax.legend(frameon=False, loc="upper left", ncol=2); style(ax)
fig.text(.5, -.02, "Bars at the varying-lead case show the mean with a 2.5th to 97.5th percentile range across 1,000 lead-time paths | " + LABEL, ha="center", fontsize=8.5, color=INK2)
plt.tight_layout(); plt.savefig(CH / "w74_02_fill_rate_constant_vs_varying.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(8.8, 5.2))
for n, sh, c, mk in zip(names, short, cols, ["o", "s", "s", "s", "D", "D", "D"]):
    ax.scatter(v.avg_stock_value_at_cost[n] / 1e6, v.fill_rate[n] * 100, s=70, color=c, marker=mk, zorder=3)
    ax.annotate(sh, (v.avg_stock_value_at_cost[n] / 1e6, v.fill_rate[n] * 100), textcoords="offset points", xytext={"B-LT 90%": (-34, 9), "B 95%": (8, -12)}.get(sh, (6, -3 if "LT" not in sh else 5)), fontsize=9, color=INK2)
ax.set_xlabel("Average stock value at cost, Rs million"); ax.set_ylabel("Fill rate, %"); ax.set_title("More fill needs more stock; the question is how much (lead time varies)"); style(ax)
fig.text(.5, -.02, "No holding or ordering cost data: stock value is a proxy, no point is called cost-optimal | " + LABEL, ha="center", fontsize=8.5, color=INK2)
plt.tight_layout(); plt.savefig(CH / "w74_03_fill_vs_stock_value.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()
check("three charts written", "3", str(len(list(CH.glob("*.png")))), len(list(CH.glob("*.png"))) == 3)

## 7. Reorder-point table and validation

In [8]:
rp = pd.DataFrame({"product_id": I.product_id, "branch_id": I.branch_id, "mean_daily_demand": mu.round(3), "sd_daily_demand": sd.round(3),
                   "lead_mean": ltm.round(2), "lead_sd": lts.round(2), "reorder_level_A": sA, "max_stock_A": SA,
                   "s_B90_mean_lead": rop_mean(1.28), "s_B95_mean_lead": rop_mean(1.645), "s_B99_mean_lead": rop_mean(2.33),
                   "s_BLT90": rop_var(1.28), "s_BLT95": rop_var(1.645), "s_BLT99": rop_var(2.33)})
rp["label"] = LABEL; rp.to_csv(OUT / "w74_reorder_points.csv", index=False)
check("B-LT trigger is at least the mean-lead trigger for every pair (it only adds lead-time variance)", "all pairs", "all pairs",
      (rp.s_BLT95 >= rp.s_B95_mean_lead - 1).all())
check("every output table carries the sales-scale label", "5 of 5", "5 of 5",
      all("label" in pd.read_csv(OUT / f).columns for f in ["w74_policy_summary.csv", "w74_pair_results.csv", "w74_comparison_vs_A.csv", "w74_pairs_below_95.csv", "w74_reorder_points.csv"]))
check("summary and pair tables carry an assumptions column", "2 of 2", "2 of 2", all("assumptions" in pd.read_csv(OUT / f).columns for f in ["w74_policy_summary.csv", "w74_pair_results.csv"]))
outs = sorted(p.name for p in OUT.glob("*.csv"))
check("output tables written (six)", "6", str(len(outs)), len(outs) == 6)
V = pd.DataFrame(checks); V.insert(0, "Check_ID", range(1, len(V) + 1)); V.to_csv("validation_report_week07_w74.csv", index=False)
print(len(V), "checks,", int((V.Status == "PASS").sum()), "pass"); assert (V.Status == "PASS").all()

30 checks, 30 pass
